# توحيد الصور والمعالجة المسبقة (Image Standardization & Preprocessing) — DMorphNet

هذا الدفتر ينفّذ خطوة **"Image Standardization and Preprocessing"** على مجموعة البيانات التي بنيناها في الدفتر الأول (45,000 صورة):

1. **توحيد الحجم**: نموذج EfficientNet-B6 يتطلب حجم إدخال ثابتاً، لذلك نغيّر حجم كل صورة إلى **528×528 بكسل**.
2. **تحسين التباين بـ CLAHE**: تقنية Contrast Limited Adaptive Histogram Equalization تحسّن تباين الصورة **دون إضافة ضوضاء**، فتصبح ملامح الوجه الدقيقة (المهمة لكشف المورف) أكثر وضوحاً.
3. **زيادة البيانات (Data Augmentation)** لإضافة تنوّع يحاكي الواقع:
   - صور **ضبابية** (Gaussian Blur).
   - صور **بضوضاء** (Gaussian Noise).
   - حفظ بضغط **JPEG بجودات مختلفة**.
   - تعديل **السطوع والتباين**.
   - **قلب** الصور أفقياً (Flip).
4. **مسار الاختبار**: تغيير الحجم ← ضغط ← **تسوية القيم (Normalization)** — بدون زيادة بيانات.
5. **نفس الخطوات تُطبَّق على الصور الحقيقية والمدموجة** بلا أي تمييز، حفاظاً على الاتساق.

> ⚠️ معالجة 45,000 صورة بحجم 528×528 تستغرق ٣٠–٦٠ دقيقة تقريباً وتحتاج ~3GB مساحة. تأكد أن جلسة Colab لديها مساحة كافية.


## ١) تثبيت المكتبات والتحقق من البيئة

معظم المكتبات (OpenCV, TensorFlow, NumPy) مثبتة مسبقاً في Colab — نثبّت الناقص فقط ونتحقق من الإصدارات.


In [ ]:
!pip -q install opencv-python-headless tqdm

import cv2, numpy as np, tensorflow as tf
print("OpenCV:", cv2.__version__)
print("TensorFlow:", tf.__version__)
print("تم تجهيز البيئة بنجاح ✅")


## ٢) تحميل مجموعة البيانات من الخطوة السابقة

نحمّل مجموعة الـ 45,000 صورة التي أنشأناها في دفتر **بناء مجموعة البيانات**:
- إذا كانت موجودة في الجلسة الحالية (`/content/DMorphNet_dataset`) نستخدمها مباشرة.
- وإلا نفكّ ضغطها من النسخة المحفوظة في Google Drive.


In [ ]:
import os, glob, shutil, random

DATA_DIR = "/content/DMorphNet_dataset"
DATASET_ZIP = "/content/drive/MyDrive/DMorphNet_dataset.zip"

if not os.path.isdir(os.path.join(DATA_DIR, "train")):
    from google.colab import drive
    drive.mount('/content/drive')
    print("جاري فك الضغط من Google Drive ...")
    shutil.unpack_archive(DATASET_ZIP, DATA_DIR)

# عرض أعداد الصور لكل قسم وفئة
SPLITS = ["train", "val", "test"]
CLASSES = ["real", "morph"]
for split in SPLITS:
    counts = {c: len(glob.glob(os.path.join(DATA_DIR, split, c, "*.jpg")))
              for c in CLASSES}
    print(f"{split}: حقيقية={counts['real']}  مدموجة={counts['morph']}  "
          f"الإجمالي={sum(counts.values())}")


## ٣) الإعدادات العامة

- حجم الإدخال **528×528** (المطلوب لـ EfficientNet-B6).
- إعدادات CLAHE: حد قصّ `clipLimit=2.0` وشبكة بلاطات 8×8 — قيم قياسية تحسّن التباين محلياً دون تضخيم الضوضاء.
- جودة JPEG عند الحفظ = 90 (خطوة الضغط الموحّدة لكل الصور).


In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

IMG_SIZE = 528                       # حجم الإدخال لـ EfficientNet-B6
CLAHE_CLIP = 2.0                     # حد القص في CLAHE (يمنع تضخيم الضوضاء)
CLAHE_GRID = (8, 8)                  # حجم شبكة البلاطات في CLAHE
JPEG_QUALITY = 90                    # جودة الضغط الموحّدة عند الحفظ

PROC_DIR = "/content/DMorphNet_processed"   # مجلد الصور بعد المعالجة
for split in SPLITS:
    for cls in CLASSES:
        os.makedirs(os.path.join(PROC_DIR, split, cls), exist_ok=True)

print(f"سيتم حفظ الصور المعالجة ({IMG_SIZE}x{IMG_SIZE} + CLAHE) في: {PROC_DIR}")


## ٤) دالة التوحيد: تغيير الحجم + CLAHE

خطوات التوحيد لكل صورة (حقيقية أو مدموجة — نفس الخطوات تماماً):
1. تغيير الحجم إلى 528×528 باستيفاء Cubic (جودة أعلى عند التكبير).
2. التحويل إلى فضاء الألوان **LAB** وتطبيق CLAHE على قناة الإضاءة **L فقط** — بهذا نحسّن التباين دون تشويه الألوان أو إضافة ضوضاء.
3. إعادة التحويل إلى BGR.

ثم نعرض مثالاً **قبل / بعد** لنرى كيف أصبحت الملامح الدقيقة أوضح.


In [ ]:
import matplotlib.pyplot as plt

clahe = cv2.createCLAHE(clipLimit=CLAHE_CLIP, tileGridSize=CLAHE_GRID)


def standardize(img_bgr):
    # 1) توحيد الحجم إلى 528x528
    img = cv2.resize(img_bgr, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_CUBIC)
    # 2) CLAHE على قناة الإضاءة L في فضاء LAB (تحسين التباين دون ضوضاء)
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    l = clahe.apply(l)
    return cv2.cvtColor(cv2.merge((l, a, b)), cv2.COLOR_LAB2BGR)


# مثال قبل/بعد على صورة عشوائية من التدريب
sample_path = random.choice(glob.glob(os.path.join(DATA_DIR, "train", "real", "*.jpg")))
orig = cv2.imread(sample_path)
proc = standardize(orig)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(cv2.cvtColor(cv2.resize(orig, (IMG_SIZE, IMG_SIZE)), cv2.COLOR_BGR2RGB))
axes[0].set_title("قبل (تغيير حجم فقط)")
axes[1].imshow(cv2.cvtColor(proc, cv2.COLOR_BGR2RGB))
axes[1].set_title("بعد (CLAHE) — تباين وملامح أوضح")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()


## ٥) تطبيق التوحيد على كل الصور (45,000 صورة)

نمرّ على جميع الأقسام (تدريب/تحقق/اختبار) وكلتا الفئتين (حقيقية/مدموجة) ونطبّق:
**تغيير الحجم ← CLAHE ← حفظ بضغط JPEG (جودة 90)**.

نفس المسار بالضبط للصور الحقيقية والمدموجة — أي اختلاف في المعالجة بين الفئتين قد يتعلمه النموذج كـ"اختصار" غير عادل.


In [ ]:
from tqdm.auto import tqdm

for split in SPLITS:
    for cls in CLASSES:
        src_dir = os.path.join(DATA_DIR, split, cls)
        dst_dir = os.path.join(PROC_DIR, split, cls)
        files = sorted(glob.glob(os.path.join(src_dir, "*.jpg")))
        for p in tqdm(files, desc=f"معالجة {split}/{cls}"):
            img = cv2.imread(p)
            if img is None:
                continue
            cv2.imwrite(os.path.join(dst_dir, os.path.basename(p)),
                        standardize(img),
                        [cv2.IMWRITE_JPEG_QUALITY, JPEG_QUALITY])

total = sum(len(glob.glob(os.path.join(PROC_DIR, s, c, "*.jpg")))
            for s in SPLITS for c in CLASSES)
print(f"✅ تمت معالجة {total} صورة (528x528 + CLAHE + ضغط JPEG)")


## ٦) دوال زيادة البيانات (Data Augmentation)

نعرّف التحويلات التي تضيف تنوّعاً واقعياً لصور **التدريب فقط** (كل تحويل يُطبّق باحتمال معيّن):

| التحويل | الاحتمال | الهدف |
|---|---|---|
| قلب أفقي (Flip) | 50% | وضعيات وجه مختلفة |
| سطوع/تباين عشوائي | 50% | إضاءات مختلفة |
| ضبابية Gaussian Blur | 30% | كاميرات منخفضة الجودة / حركة |
| ضوضاء Gaussian Noise | 30% | حساسات ضعيفة وإضاءة منخفضة |
| إعادة ضغط JPEG (جودة 40–90) | 40% | صور مرسلة/مضغوطة كما في الواقع |

بهذا يتعامل النموذج مع أنواع الصور المختلفة في السيناريوهات الحقيقية، ولا يعتمد على تفاصيل مثالية غير موجودة عملياً.


In [ ]:
def augment(img):
    # 1) قلب أفقي
    if random.random() < 0.5:
        img = cv2.flip(img, 1)
    # 2) تعديل السطوع والتباين
    if random.random() < 0.5:
        alpha = random.uniform(0.8, 1.2)      # التباين
        beta = random.uniform(-25, 25)        # السطوع
        img = cv2.convertScaleAbs(img, alpha=alpha, beta=beta)
    # 3) ضبابية
    if random.random() < 0.3:
        k = random.choice([3, 5, 7])
        img = cv2.GaussianBlur(img, (k, k), 0)
    # 4) ضوضاء غاوسية
    if random.random() < 0.3:
        sigma = random.uniform(5, 15)
        noise = np.random.normal(0, sigma, img.shape)
        img = np.clip(img.astype(np.float32) + noise, 0, 255).astype(np.uint8)
    # 5) إعادة ضغط JPEG بجودة عشوائية
    if random.random() < 0.4:
        q = random.randint(40, 90)
        ok, enc = cv2.imencode(".jpg", img, [cv2.IMWRITE_JPEG_QUALITY, q])
        if ok:
            img = cv2.imdecode(enc, cv2.IMREAD_COLOR)
    return img


# عرض أمثلة: صورة معالجة وخمس نسخ معزّزة منها
sample = cv2.imread(random.choice(
    glob.glob(os.path.join(PROC_DIR, "train", "morph", "*.jpg"))))

fig, axes = plt.subplots(1, 6, figsize=(20, 4))
axes[0].imshow(cv2.cvtColor(sample, cv2.COLOR_BGR2RGB))
axes[0].set_title("الأصل (بعد التوحيد)")
for i in range(1, 6):
    axes[i].imshow(cv2.cvtColor(augment(sample.copy()), cv2.COLOR_BGR2RGB))
    axes[i].set_title(f"نسخة معزّزة {i}")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()


## ٧) بناء خطوط الإدخال (tf.data) لـ EfficientNet-B6

نبني ثلاثة مسارات جاهزة للتدريب:

- **التدريب**: قراءة ← زيادة بيانات (عشوائية في كل حقبة Epoch) ← **تسوية** إلى المدى [0,1] ← خلط ← دفعات.
- **التحقق والاختبار**: قراءة ← تسوية ← دفعات فقط — *(الصور خضعت مسبقاً لتغيير الحجم والضغط في خطوة التوحيد، فيتبقى التسوية فقط — تماماً كما في الورقة: تغيير حجم ← ضغط ← تسوية)*.

الملصقات: `0 = حقيقية`، `1 = مدموجة`. ونفس المسار يُطبّق على الفئتين معاً.


In [ ]:
BATCH_SIZE = 8          # حجم الدفعة (528x528 صور كبيرة — قلّل إذا امتلأت الذاكرة)
AUTOTUNE = tf.data.AUTOTUNE

LABEL_MAP = {"real": 0, "morph": 1}


def list_files(split):
    paths, labels = [], []
    for cls, lab in LABEL_MAP.items():
        fs = sorted(glob.glob(os.path.join(PROC_DIR, split, cls, "*.jpg")))
        paths += fs
        labels += [lab] * len(fs)
    return paths, labels


def decode(path, label):
    # قراءة الصورة وفك ترميز JPEG
    img = tf.io.decode_jpeg(tf.io.read_file(path), channels=3)
    img = tf.cast(img, tf.float32)
    img.set_shape([IMG_SIZE, IMG_SIZE, 3])
    return img, label


def tf_augment(img, label):
    # تنفيذ دوال الزيادة (OpenCV) داخل خط tf.data
    def _aug(x):
        x = augment(x.astype(np.uint8)[:, :, ::-1])   # RGB -> BGR للـ OpenCV
        return x[:, :, ::-1].astype(np.float32)        # ثم العودة إلى RGB
    img = tf.numpy_function(_aug, [img], tf.float32)
    img.set_shape([IMG_SIZE, IMG_SIZE, 3])
    return img, label


def normalize(img, label):
    # التسوية إلى المدى [0,1]
    return img / 255.0, label


def make_dataset(split, training=False):
    paths, labels = list_files(split)
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        ds = ds.shuffle(len(paths), seed=SEED)
    ds = ds.map(decode, num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.map(tf_augment, num_parallel_calls=AUTOTUNE)
    ds = ds.map(normalize, num_parallel_calls=AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)


train_ds = make_dataset("train", training=True)
val_ds   = make_dataset("val")
test_ds  = make_dataset("test")
print("✅ خطوط الإدخال جاهزة لتغذية EfficientNet-B6")


## ٨) التحقق من خطوط الإدخال

نتأكد أن كل شيء صحيح قبل التدريب:
- شكل الدفعة يجب أن يكون `(8, 528, 528, 3)`.
- قيم البكسل بعد التسوية داخل المدى `[0, 1]`.
- الدفعة تحتوي الفئتين (حقيقية ومدموجة)، ونعرض عيّنة منها.


In [ ]:
images, labels = next(iter(train_ds))
print("شكل دفعة الصور:", images.shape)
print("شكل دفعة الملصقات:", labels.shape)
print(f"مدى قيم البكسل: [{float(tf.reduce_min(images)):.3f}, "
      f"{float(tf.reduce_max(images)):.3f}]")
assert images.shape[1:] == (IMG_SIZE, IMG_SIZE, 3), "حجم الصور غير صحيح!"
assert float(tf.reduce_min(images)) >= 0.0 and float(tf.reduce_max(images)) <= 1.0

fig, axes = plt.subplots(1, min(6, len(images)), figsize=(18, 4))
names = {0: "حقيقية", 1: "مدموجة"}
for i, ax in enumerate(np.ravel(axes)):
    ax.imshow(images[i].numpy())
    ax.set_title(names[int(labels[i])])
    ax.axis("off")
plt.tight_layout()
plt.show()
print("✅ خطوط الإدخال سليمة وجاهزة لتدريب DMorphNet")


## ٩) (اختياري) حفظ الصور المعالجة في Google Drive

نضغط مجلد الصور المعالجة (528×528 + CLAHE) ونحفظه في Drive حتى نبدأ منه مباشرة في دفتر التدريب دون إعادة المعالجة.

> الحجم التقريبي ~3GB — تأكد من توفر مساحة كافية في Drive.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

archive = shutil.make_archive("/content/DMorphNet_processed", "zip", PROC_DIR)
print("تم إنشاء الأرشيف:", archive)

dest = "/content/drive/MyDrive/DMorphNet_processed.zip"
shutil.copy(archive, dest)
print("تم الحفظ في Google Drive:", dest)
